# Espiral de manchas solares (Sunspot Spiral) — dos vistas

Este notebook genera **dos animaciones** a partir de tus datos reales de manchas solares (`SN_d_tot_V2.0.csv`, SILSO/WDC-SILSO):

1. **`sunspot_spiral_top.gif`** — vista polar desde el eje Z (como el inicio del reel), con los
   meses rotulados en el plano polar.
2. **`sunspot_spiral_vertical.gif`** — vista vertical/inclinada en 3D, con una escala de
   referencia fija (μ, 1σ, 2σ, 3σ) a los lados y etiquetas de década/ciclo solar fijas en pantalla.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from mpl_toolkits.mplot3d import proj3d

plt.style.use('dark_background')


## 1. Carga y limpieza de datos

Cargamos el CSV real (mismo formato que trae por defecto: separador `;`, sin encabezado), construimos una
columna `Date` propiamente tipada, y convertimos los centinelas `-1` en `NaN` en vez de tratarlos
como manchas solares negativas. `interpolate(limit=...)` rellena huecos cortos (unos pocos días
sin reporte) sin inventar tendencias en huecos largos.

Los datos provienen de la fuente original de SILSO: https://www.sidc.be/SILSO/datafiles

In [2]:
CSV_PATH = "SN_d_tot_V2.0.csv"

names = ["Year", "Month", "Day", "Decimal date",
         "Estimated Sunspot Number", "Estimated Standard Deviation",
         "Number of Stations calculated", "Number of Stations available"]

raw = pd.read_csv(CSV_PATH, delimiter=";", names=names)
raw["Date"] = pd.to_datetime(dict(year=raw.Year, month=raw.Month, day=raw.Day))

# -1 es el centinela SILSO de "sin observación": lo convertimos a NaN, no a "cero manchas".
raw.loc[raw["Estimated Sunspot Number"] < 0, "Estimated Sunspot Number"] = np.nan
raw.loc[raw["Estimated Standard Deviation"] < 0, "Estimated Standard Deviation"] = np.nan

print(f"Rango disponible: {raw.Date.min().date()} -> {raw.Date.max().date()}")
print(f"Días sin observación (NaN tras limpieza): {raw['Estimated Sunspot Number'].isna().sum()}")


Rango disponible: 1818-01-01 -> 2026-08-31
Días sin observación (NaN tras limpieza): 3247


## 2. Ventana temporal, remuestreo y suavizado

Animar 208 años día a día (76214 puntos) es pesado y, sobre todo antes de 1849, mayormente vacío.
Definimos una ventana (`DATE_START`/`DATE_END`), un remuestreo (`RESAMPLE`, p. ej. `"3D"` = medias
cada 3 días) y un suavizado móvil corto (`SMOOTH_WIN`) que sigue el espíritu del número de manchas
"suavizado" que usa la comunidad heliofísica (evita que la espiral tiemble día a día por ruido de
conteo), sin ocultar los máximos y mínimos de ciclo.

Cambiar `DATE_START` a `"1850-01-01"` con la serie histórica completa; con `RESAMPLE` más
grueso (p. ej. `"7D"`) es manejable, pero tomará más tiempo generar el GIF.

In [3]:
DATE_START = "1850-01-01"   # inicio de ventana a animar (ajustable)
DATE_END = raw["Date"].max().strftime("%Y-%m-%d")
RESAMPLE = "3D"             # cadencia de remuestreo: "1D" (diario), "3D", "7D" (semanal)...
SMOOTH_WIN = 9               # ventana de suavizado móvil, en pasos de RESAMPLE

window = raw[(raw.Date >= DATE_START) & (raw.Date <= DATE_END)].copy()
window = window.set_index("Date").sort_index()
window["SSN"] = window["Estimated Sunspot Number"].interpolate(limit=10)

df = window["SSN"].resample(RESAMPLE).mean().to_frame("SSN")
df["SSN_smooth"] = df["SSN"].rolling(SMOOTH_WIN, center=True, min_periods=1).mean()
df = df.dropna(subset=["SSN_smooth"]).reset_index()

df["Year"] = df["Date"].dt.year
df["DayOfYear"] = df["Date"].dt.dayofyear
df["TotalDays"] = df["Date"].dt.is_leap_year.map({True: 366, False: 365})

print(f"Puntos a animar: {len(df)}  (de {DATE_START} a {DATE_END}, remuestreo {RESAMPLE})")
df.head()


Puntos a animar: 21509  (de 1850-01-01 a 2026-08-31, remuestreo 3D)


,Date,SSN,SSN_smooth,Year,DayOfYear,TotalDays
0,1850-01-01,210.666667,145.266667,1850,1,365
1,1850-01-04,136.666667,142.000000,1850,4,365
2,1850-01-07,104.333333,134.476190,1850,7,365
3,1850-01-10,158.000000,131.458333,1850,10,365
4,1850-01-13,116.666667,144.222222,1850,13,365


## 3. Geometría, colores y anillos de referencia

- `Theta`: ángulo según el día del año (una vuelta completa = un año calendario).
- `Zyear`: altura continua = años transcurridos desde el inicio de la ventana + fracción del año
  — es lo que convierte la espiral plana en una hélice ascendente para la vista vertical.
- `Radius`: base fija (`RADIO_BASE`) más el número de manchas suavizado, escalado por `K`. Los
  niveles de referencia (μ, μ+1σ, μ+2σ, μ+3σ) se calculan con la misma fórmula para que sean
  directamente comparables con el radio de la línea.

`CMAP` y el `Normalize` de color se definen aquí, una sola vez, y las dos funciones de animación
los reciben ya hechos — así se garantiza que ambas vistas usen exactamente la misma paleta y la
misma escala de color.

In [4]:
RADIO_BASE = 5.0
K = 0.03  # escala de manchas solares -> unidades de radio

df["Theta"] = (df["DayOfYear"] - 1) / df["TotalDays"] * 2 * np.pi
min_year = df["Year"].min()
df["Zyear"] = (df["Year"] - min_year) + df["DayOfYear"] / df["TotalDays"]

df["Radius"] = RADIO_BASE + df["SSN_smooth"] * K
df["X"] = df["Radius"] * np.cos(df["Theta"])
df["Y"] = df["Radius"] * np.sin(df["Theta"])

mu = df["SSN_smooth"].mean()
sigma = df["SSN_smooth"].std()
sigma_levels = [1, 2, 3]
ring_radii = {n: RADIO_BASE + max(mu + n * sigma, 0) * K for n in sigma_levels}
mean_radius = RADIO_BASE + mu * K

print(f"mu (SSN suavizado, ventana graficada) = {mu:.1f}")
print(f"sigma = {sigma:.1f}")
print("Radios de referencia:", {f"{n}sigma": round(r, 2) for n, r in ring_radii.items()})

vmax_color = df["SSN_smooth"].max()
month_labels = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
month_angles = [(pd.Timestamp(2001, m, 15).dayofyear - 1) / 365 * 2 * np.pi for m in range(1, 13)]

# --- paleta y estilo de referencia compartidos por ambas vistas ---
CMAP = "plasma"
COLOR_NORM = Normalize(vmin=0, vmax=vmax_color)
RING_COLORS = {1: "#67e8f9", 2: "#fde047", 3: "#fb7185"}  # cian, amarillo, rojo-rosado
MEAN_COLOR = "#f97316"  # naranja

def make_legend_handles():
    return [
        Line2D([0], [0], color=MEAN_COLOR, lw=1.2, label="\u03bc (mean avg.)"),
        Line2D([0], [0], color=RING_COLORS[1], lw=1.2, ls="--", label="1\u03c3"),
        Line2D([0], [0], color=RING_COLORS[2], lw=1.2, ls="--", label="2\u03c3"),
        Line2D([0], [0], color=RING_COLORS[3], lw=1.2, ls="--", label="3\u03c3"),
    ]


mu (SSN suavizado, ventana graficada) = 84.0
sigma = 67.8
Radios de referencia: {'1sigma': np.float64(9.55), '2sigma': np.float64(11.59), '3sigma': np.float64(13.62)}


## 3b. Etiquetas de década / ciclo solar

Para la vista vertical, generamos una lista de marcas `(altura_Z, texto)` a colocar como
etiquetas con recuadro, se fija en pantalla, a la altura que les corresponde. `LABEL_MODE =
"decade"` coloca una etiqueta al empezar cada década (2010, 2020, ...); `LABEL_MODE = "cycle"`
usa en cambio las fechas de inicio de los ciclos solares 21–25 (SIDC) que caigan dentro de tu
ventana `DATE_START`/`DATE_END`. Con la ventana por defecto (2008 en adelante) el modo "decade"
marca 2010 y 2020, y el modo "cycle" marca el inicio de los ciclos 24 y 25.

In [5]:
LABEL_MODE = "decade"  # "decade" o "cycle"

SOLAR_CYCLE_STARTS = {  # inicios de ciclo solar aproximados (SIDC), ciclos 21-25
                        # https://en.wikipedia.org/wiki/List_of_solar_cycles
    9: "1843-07-01", 10: "1855-12-01", 11: "1867-03-01", 
    12: "1878-12-01", 13: "1890-03-01", 14: "1902-01-01", 
    15: "1913-07-01", 16: "1923-08-01", 17: "1933-10-01", 
    18: "1944-02-01", 19: "1954-04-01", 20: "1964-10-01", 
    21: "1976-03-01", 22: "1986-09-01", 23: "1996-08-01",
    24: "2008-12-01", 25: "2019-12-01",
}

def _zyear_for_date(date):
    date = pd.Timestamp(date)
    total_days = 366 if date.is_leap_year else 365
    return (date.year - min_year) + (date.dayofyear - 1) / total_days

time_labels = []
if LABEL_MODE == "decade":
    for year in range(int(df["Year"].min()), int(df["Year"].max()) + 1):
        if year % 10 == 0:
            time_labels.append((_zyear_for_date(f"{year}-01-01"), str(year)))
elif LABEL_MODE == "cycle":
    for cycle, start in SOLAR_CYCLE_STARTS.items():
        start_ts = pd.Timestamp(start)
        if df["Date"].min() <= start_ts <= df["Date"].max():
            time_labels.append((_zyear_for_date(start_ts), f"Ciclo {cycle}"))

print("Etiquetas verticales:", time_labels)


Etiquetas verticales: [(np.float64(0.0), '1850'), (np.float64(10.0), '1860'), (np.float64(20.0), '1870'), (np.float64(30.0), '1880'), (np.float64(40.0), '1890'), (np.float64(50.0), '1900'), (np.float64(60.0), '1910'), (np.float64(70.0), '1920'), (np.float64(80.0), '1930'), (np.float64(90.0), '1940'), (np.float64(100.0), '1950'), (np.float64(110.0), '1960'), (np.float64(120.0), '1970'), (np.float64(130.0), '1980'), (np.float64(140.0), '1990'), (np.float64(150.0), '2000'), (np.float64(160.0), '2010'), (np.float64(170.0), '2020')]


## 4. Versión 1 — vista polar desde el eje Z (con meses)

Esta es la vista "de reloj" con la que arranca el reel: cámara mirando directo hacia abajo por el
eje Z, de modo que solo importan `(Theta, Radius)`. En vez de simular esa vista con un `Axes3D` y
`elev=90`, usamos directamente una proyección `polar` de matplotlib — es más nítida, más rápida de
animar, y `set_xticklabels` coloca los meses exactamente en el borde del círculo sin distorsión de
perspectiva.

Los niveles μ/1σ/2σ/3σ tienen cada uno su color (ver leyenda debajo del gráfico) y ahora son más
finos y translúcidos, para que no compitan visualmente con la espiral. El trazo de la espiral es
una línea sólida coloreada (`LineCollection`): cada segmento toma el color del número de manchas
promedio entre sus dos extremos.


In [6]:
def build_top_view(df, step_size=4, fps=15, out_path="sunspot_spiral_top.gif"):
    fig = plt.figure(figsize=(6.0, 7.5), facecolor="black", dpi=200)
    fig.subplots_adjust(bottom=0.09, top=0.90)
    ax = fig.add_subplot(111, projection="polar")
    ax.set_facecolor("black")
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_ylim(0, max(ring_radii.values()) * 1.2)
    ax.set_xticks(month_angles)
    ax.set_xticklabels(month_labels, color="white", fontsize=10)
    ax.set_yticklabels([])
    ax.grid(color="gray", alpha=0.25)
    ax.spines['polar'].set_color("gray")

    theta_full = np.linspace(0, 2 * np.pi, 200)
    ax.plot(theta_full, [mean_radius] * 200, ls="-", lw=1.0, color=MEAN_COLOR, alpha=0.55, zorder=1)
    for n, r in ring_radii.items():
        ax.plot(theta_full, [r] * 200, ls="--", lw=0.9, color=RING_COLORS[n], alpha=0.45, zorder=1)

    lc = LineCollection([], cmap=CMAP, norm=COLOR_NORM, linewidths=2.2, zorder=2)
    lc.set_cmap(CMAP)
    ax.add_collection(lc)

    title_obj = ax.text(0.5, 1.14, "", transform=ax.transAxes, ha="center",
                         fontsize=14, color="white", weight="bold")
    cbar = fig.colorbar(lc, ax=ax, pad=0.1, shrink=0.6)
    cbar.set_label("Sunspot number (smoothed)", color="white")
    cbar.ax.yaxis.set_tick_params(color="white")
    plt.setp(plt.getp(cbar.ax, "yticklabels"), color="white")

    fig.legend(handles=make_legend_handles(), loc="lower center", ncol=4, frameon=False,
               labelcolor="white", bbox_to_anchor=(0.5, 0.015), fontsize=12,
               handlelength=1.4, handletextpad=0.5, columnspacing=1.0)

    total_frames = max(len(df) // step_size, 1)

    def update(frame):
        idx = max(min((frame + 1) * step_size, len(df)), 2)
        theta = df["Theta"].iloc[:idx].values
        radius = df["Radius"].iloc[:idx].values
        colorvals = df["SSN_smooth"].iloc[:idx].values
        points = np.column_stack([theta, radius])[:, None, :]
        segments = np.concatenate([points[:-1], points[1:]], axis=1)
        lc.set_segments(segments)
        lc.set_array((colorvals[:-1] + colorvals[1:]) / 2)
        row = df.iloc[idx - 1]
        title_obj.set_text(f"{row['Date'].strftime('%Y-%m')}  |  SSN smoothed: {row['SSN_smooth']:.0f}")
        return lc, title_obj

    ani = animation.FuncAnimation(fig, update, frames=total_frames, interval=1000 / fps, blit=False)
    Writer = animation.FFMpegWriter(fps=20, bitrate=4000)
    ani.save("sunspot_spiral_top.mp4", writer=Writer)
    #ani.save(out_path, writer="pillow", fps=fps)
    plt.close(fig)
    print(f"Guardado: {out_path}  ({total_frames} frames)")
    return out_path


## 5. Versión 2 — vista vertical/inclinada en 3D, con escala fija y etiquetas fijas

Conservamos el `Axes3D` de tu código original (la hélice ascendente solo tiene sentido en 3D) y
la cámara sigue rotando lentamente como en las revisiones anteriores. Lo que cambia:

- En vez de ~24 anillos horizontales punteados (uno por altura × nivel de σ), hay **8 líneas
  verticales fijas** (4 a cada lado: μ, 1σ, 2σ, 3σ), al estilo de la espiral climática que inspira
  el reel. **Corrección importante sobre la revisión anterior:** esas 8 líneas se ven fijas en
  pantalla, pero su posición horizontal ya no es arbitraria — se calcula proyectando el círculo
  real de cada radio (μ, 1σ, 2σ, 3σ) con la matriz de cámara de matplotlib (`ax.get_proj()` +
  `proj3d.proj_transform`) para encontrar exactamente dónde cae, en pantalla, el borde de ese
  anillo. Un anillo centrado en el eje Z se ve igual de ancho sin importar el `azim` de la cámara
  (solo cambia qué punto del anillo queda en el borde, no cuán ancho se ve el anillo) — por eso
  esa posición se calcula una sola vez, antes de animar (`radius_screen_fractions`), y se puede
  usar como una escala fija sin recalcularla en cada frame. Con esto, la espiral sí cruza visualmente
  cada línea cuando el número de manchas supera ese umbral, igual que en la vista polar.
- Las etiquetas de década/ciclo (celda 3b) también están ancladas en `ax.transAxes`, así que no
  "giran" con la cámara. **Segunda corrección:** su posición vertical usaba antes un reparto
  lineal ingenuo de `Zyear` entre el borde inferior y superior del eje, que no correspondía a
  dónde cae realmente esa altura en la proyección 3D (por eso el año de la etiqueta no coincidía
  con la parte de la hélice que le corresponde). Ahora se calcula igual que la escala de σ:
  proyectando el punto `(0, 0, Zyear)` — sobre el propio eje de la hélice, por lo que también es
  invariante al `azim` — con la matriz de cámara real (`z_screen_fractions`).
- Sigue habiendo una leyenda de color debajo del gráfico (igual que en la vista polar), y la línea
  de la espiral sigue siendo un trazo sólido coloreado (`Line3DCollection`).


In [7]:
def radius_screen_fractions(ax, radii, elev=18, z_ref=None):
    """Para cada radio en `radii`, calcula en que fraccion del eje (ax.transAxes)
    cae su punto mas a la izquierda / mas a la derecha, proyectando el circulo
    real de ese radio con la matriz de camara actual (elev, azim=0 de referencia).
    Este resultado NO depende del azim (un anillo centrado en Z se ve igual de
    ancho se mire desde donde se mire, solo cambia QUE punto del anillo queda en
    el borde) -- por eso podemos calcularlo una sola vez, antes de animar, y usarlo
    como una escala fija en pantalla que si coincide con el radio real de los datos.
    """
    if z_ref is None:
        z_ref = (df["Zyear"].min() + df["Zyear"].max()) / 2
    prev_elev, prev_azim = ax.elev, ax.azim
    ax.view_init(elev=elev, azim=0)
    proj = ax.get_proj()
    thetas = np.linspace(0, 2 * np.pi, 721)
    fractions = {}
    for level, r in radii.items():
        xs, ys = r * np.cos(thetas), r * np.sin(thetas)
        zs = np.full_like(thetas, z_ref)
        x2d, y2d, _ = proj3d.proj_transform(xs, ys, zs, proj)
        i_right, i_left = np.argmax(x2d), np.argmin(x2d)
        disp_right = ax.transData.transform((x2d[i_right], y2d[i_right]))
        disp_left = ax.transData.transform((x2d[i_left], y2d[i_left]))
        frac_right = ax.transAxes.inverted().transform(disp_right)[0]
        frac_left = ax.transAxes.inverted().transform(disp_left)[0]
        fractions[level] = (frac_left, frac_right)
    ax.view_init(elev=prev_elev, azim=prev_azim)
    return fractions


def draw_side_scale(fig, ax):
    """Escala de referencia fija (mu, 1s, 2s, 3s) a ambos lados, como superposicion 2D,
    posicionada en el radio REAL de cada nivel (ver radius_screen_fractions) -- asi la
    espiral efectivamente cruza estas lineas cuando el numero de manchas supera cada
    umbral, igual que pasa con los anillos de la vista polar."""
    levels = ["mean", 1, 2, 3]
    colors = {"mean": MEAN_COLOR, 1: RING_COLORS[1], 2: RING_COLORS[2], 3: RING_COLORS[3]}
    labels = {"mean": "\u03bc", 1: "1\u03c3", 2: "2\u03c3", 3: "3\u03c3"}
    radii = {"mean": mean_radius, 1: ring_radii[1], 2: ring_radii[2], 3: ring_radii[3]}

    fractions = radius_screen_fractions(ax, radii)
    zmin, zmax = df["Zyear"].min(), df["Zyear"].max()
    z_fracs = z_screen_fractions(ax, [zmin, zmax])
    y0, y1 = z_fracs[zmin], z_fracs[zmax]

    for level in levels:
        frac_left, frac_right = fractions[level]
        for x in (frac_left, frac_right):
            line = Line2D([x, x], [y0, y1], transform=ax.transAxes, color=colors[level],
                          lw=1.4, alpha=0.65, ls="-" if level == "mean" else "--", zorder=5)
            fig.add_artist(line)
        ax.text2D(frac_right, y1 + 0.015, labels[level], transform=ax.transAxes, color=colors[level],
                fontsize=8.5, ha="center", va="bottom", zorder=5)
        ax.text2D(frac_left, y1 + 0.015, labels[level], transform=ax.transAxes, color=colors[level],
                fontsize=8.5, ha="center", va="bottom", zorder=5)


def z_screen_fractions(ax, zvalues, elev=18):
    """Para cada altura Z en `zvalues`, calcula la fraccion vertical (ax.transAxes)
    donde realmente cae en pantalla, proyectando el punto (x=0, y=0, z) -- es decir,
    un punto sobre el propio eje de la helice -- con la matriz de camara actual.
    Un punto con x=y=0 es invariante al azim (rotar la camara alrededor del eje Z no
    lo mueve), asi que, igual que con radius_screen_fractions, esto se calcula una
    sola vez antes de animar y sigue siendo valido en todos los frames.

    OJO: la version anterior asumia que Zyear se reparte linealmente entre el borde
    inferior y superior del eje (y0=0.08 a y1=0.86 de ax.transAxes). Esa suposicion
    era incorrecta -- la proyeccion 3D no es lineal en Z de esa forma simple -- y por
    eso las etiquetas de decada/ciclo no coincidian con la altura real de la espiral.
    Con la proyeccion real, el año que marca cada etiqueta si concuerda con donde esta
    esa parte de la helice en pantalla.
    """
    prev_elev, prev_azim = ax.elev, ax.azim
    ax.view_init(elev=elev, azim=0)
    proj = ax.get_proj()
    fractions = {}
    for z in zvalues:
        x2d, y2d, _ = proj3d.proj_transform([0.0], [0.0], [z], proj)
        disp = ax.transData.transform((x2d[0], y2d[0]))
        fractions[z] = ax.transAxes.inverted().transform(disp)[1]
    ax.view_init(elev=prev_elev, azim=prev_azim)
    return fractions


def draw_time_labels(fig, ax, x=0.72):
    """Etiquetas de decada/ciclo, fijas en pantalla (no giran con la camara) y alineadas
    con la altura real (Zyear) que representan en la helice."""
    zvalues = [zyear for zyear, _ in time_labels]
    y_fracs = z_screen_fractions(ax, zvalues)
    for zyear, text in time_labels:
        y = y_fracs[zyear]
        ax.text2D(x, y, text, transform=ax.transAxes, color="white", fontsize=9,
                ha="left", va="center", zorder=5,
                bbox=dict(boxstyle="round,pad=0.25", facecolor="black", edgecolor="gray", alpha=0.75))


def build_vertical_view(df, step_size=5, fps=30, out_path="sunspot_spiral_vertical.gif"):
    fig = plt.figure(figsize=(6.0, 7.5), facecolor="black", dpi=200)
    fig.subplots_adjust(bottom=0.05, top=0.92, left=0.02, right=0.98)
    ax = fig.add_subplot(111, projection="3d")
    ax.set_facecolor("black")
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.set_axis_off()

    max_extent = max(df["Radius"].max(), max(ring_radii.values())) + 0.5
    ax.set_xlim(-max_extent, max_extent)
    ax.set_ylim(-max_extent, max_extent)
    ax.set_zlim(df["Zyear"].min(), df["Zyear"].max())
    
    ax.view_init(elev=18, azim=0)  # vista de referencia para fijar limites/proyeccion inicial
    ax.set_box_aspect((1, 1, 1.6))  # (x, y, z) -- sube el último número para alargar el eje Z

    draw_side_scale(fig, ax)
    draw_time_labels(fig, ax)

    lc = Line3DCollection([], cmap=CMAP, norm=COLOR_NORM, linewidths=2.2, alpha=0.95)
    lc.set_cmap(CMAP)
    ax.add_collection3d(lc, autolim=False)

    title_obj = ax.text2D(0.5, 0.88, "", transform=fig.transFigure, ha="center",
                           fontsize=15, color="white", weight="bold")
    fig.legend(handles=make_legend_handles(), loc="lower center", ncol=4, frameon=False,
               labelcolor="white", bbox_to_anchor=(0.5, 0.02), fontsize=12,
               handlelength=1.4, handletextpad=0.5, columnspacing=1.0)

    total_frames = max(len(df) // step_size, 1)

    def update(frame):
        idx = max(min((frame + 1) * step_size, len(df)), 2)
        x = df["X"].iloc[:idx].values
        y = df["Y"].iloc[:idx].values
        z = df["Zyear"].iloc[:idx].values
        colorvals = df["SSN_smooth"].iloc[:idx].values
        points = np.array([x, y, z]).T.reshape(-1, 1, 3)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)
        lc.set_segments(segments)
        lc.set_array((colorvals[:-1] + colorvals[1:]) / 2)
        ax.view_init(elev=18, azim=frame * 0.8)
        row = df.iloc[idx - 1]
        title_obj.set_text(f"{row['Date'].strftime('%Y-%m')}  |  SSN smoothed: {row['SSN_smooth']:.0f}")
        return lc, title_obj

    ani = animation.FuncAnimation(fig, update, frames=total_frames, interval=1000 / fps, blit=False)
    Writer = animation.FFMpegWriter(fps=20, bitrate=4000)
    ani.save("sunspot_spiral_vertical.mp4", writer=Writer)
    #ani.save(out_path, writer="pillow", fps=fps)
    plt.close(fig)
    print(f"Guardado: {out_path}  ({total_frames} frames)")
    return out_path


## 6. Generar ambos GIFs

`step_size` controla cuántos puntos de datos avanza cada frame (mayor = video más corto/rápido de
renderizar, menor = más fluido y más lento de generar). Con la ventana y remuestreo por defecto
(`2008-01-01` en adelante, cada 3 días) cada GIF pesa unos 7-20 MB y tarda del orden de 30-90
segundos en generarse; ajustar `DATE_START`/`RESAMPLE`/`step_size`/`fps` en las celdas 3 y 6 si
se quiere más o menos detalle (a costa de un archivo más pesado y más lento de renderizar).

Si se tiene `ffmpeg` instalado, `animation.FFMpegWriter` exporta a `.mp4` bastante más rápido que
`pillow` a `.gif`, con mejor calidad de color (el GIF cuantiza la paleta a 256 colores). Se puede
generar el `.mp4` y convertirlo a `.gif` después solo si de verdad se necesita el formato GIF para
web o redes.

In [8]:
top_path = build_top_view(df, step_size=20, fps=50)

Guardado: sunspot_spiral_top.gif  (1075 frames)


In [9]:
vertical_path = build_vertical_view(df, step_size=20, fps=50)

Guardado: sunspot_spiral_vertical.gif  (1075 frames)


## 7. (Opcional) Exportar a MP4 con ffmpeg — más rápido, mejor color

Descomentar y ejecutar si se tiene `ffmpeg` disponible en máquina. Es el mismo patrón de
`build_top_view`/`build_vertical_view`; solo cambia el `writer`.

In [10]:
#Writer = animation.FFMpegWriter(fps=20, bitrate=4000)
#ani.save("sunspot_spiral_top.mp4", writer=Writer)
